# Set Up

## A. Library

Menginstall Library: (1) Pytorch dan (2) Transformer

In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
%pip install transformers

Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Mengimport Library

In [2]:
import os
import torch
import logging
import warnings
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.patches as mpatches

d:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\env_ta\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## B. Device

Melakukan Set Up Device:<br>(1) Mendeteksi Hardware,<br>(2) Melakukan Konfigurasi Device yang Akan Digunakan Dalam Eksperimen ini,<br>(3) Memastikan Komputasi Berjalan di GPU

In [9]:
# =========================================================
# 1. SETUP LIBRARY & DEVICE
# =========================================================

print("-" * 50)
# Logika pendeteksian perangkat
# Jika CUDA (GPU NVIDIA) tersedia, gunakan 'cuda'. Jika tidak, pakai 'cpu'.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device yang digunakan: {device}")

if torch.cuda.is_available():
    print(f"Nama GPU: {torch.cuda.get_device_name(0)}")
    print(f"Versi CUDA: {torch.version.cuda}")
else:
    print("Peringatan: GPU tidak terdeteksi. Proses akan berjalan lambat di CPU.")
print("-" * 50)

os.environ["DISABLE_SAFETENSORS_CONVERSION"] = "1"

--------------------------------------------------
Device yang digunakan: cuda
Nama GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Versi CUDA: 12.4
--------------------------------------------------


## C. Model

### 1. Load Model IndoBERT (Versi IndoLEM)

Load Model IndoBERT Versi IndoLEM (Koto et al., 2020)

In [10]:
print("Melakukan Loading Model IndoBERT (IndoLEM):")
print("="*75)
indobert_name = "indolem/indobert-base-uncased"
tokenizer_indobert = AutoTokenizer.from_pretrained(indobert_name)
model_indobert = AutoModel.from_pretrained(indobert_name, output_hidden_states=True).to(device)
model_indobert.eval() # Set ke mode evaluasi agar setiap dijalankan tidak terjadi perubahan pada bobot model
print("Info: Model IndoBERT berhasil dimuat✅")
print("="*75)

Melakukan Loading Model IndoBERT (IndoLEM):


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11908.16it/s]
BertModel LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Info: Model IndoBERT berhasil dimuat✅


Note: Model yang Diload Menggunakan Model Evaluasi Dikarnakan Kita Hanya Ingin Mengekstrak Representasinya Saja Sehingga Komponen Dropout-nya Perlu untuk Dimatikan Agar Representasi yang Keluar dari Tiap Lapisan Tidak Akan Berubah-ubah Setiap Kali Dijalankan

### 2. Load Model XLM-R

In [11]:
print("Melakukan Loading Model XLM-R:")
print("="*75)
xlmr_name = "xlm-roberta-base"
tokenizer_xlmr = AutoTokenizer.from_pretrained(xlmr_name)
model_xlmr = AutoModel.from_pretrained(xlmr_name, output_hidden_states=True).to(device)
model_xlmr.eval() # Set ke mode evaluasi agar setiap dijalankan tidak terjadi perubahan pada bobot model
print("Info: Model XLM-R berhasil dimuat✅")
print("="*75)

Melakukan Loading Model XLM-R:


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2078.84it/s]
XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Info: Model XLM-R berhasil dimuat✅


Note: Model yang Diload Menggunakan Model Evaluasi Dikarnakan Kita Hanya Ingin Mengekstrak Representasinya Saja Sehingga Komponen Dropout-nya Perlu untuk Dimatikan Agar Representasi yang Keluar dari Tiap Lapisan Tidak Akan Berubah-ubah Setiap Kali Dijalankan

# Eksperimen

## A. Morfologi Afiksasi (Surface)

### 1. Load Dataset dan Encode Label

Encode label: Konfiks = 0, Prefiks = 1, Sufiks = 2, Tidak Berimbuhan = 3

In [15]:
# ============================================================
# TAHAP 1: LOAD DATASET MORFOLOGI AFIKSASI
# ============================================================

BASE_PROCESSED = r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\data\processed\Morfologi_Afiksasi"

df_train_raw = pd.read_csv(os.path.join(BASE_PROCESSED, "ud-indo-gsd_train_morph.csv"))
df_dev_raw   = pd.read_csv(os.path.join(BASE_PROCESSED, "ud-indo-gsd_dev_morph.csv"))
df_test_raw  = pd.read_csv(os.path.join(BASE_PROCESSED, "ud-indo-gsd_test_morph.csv"))

COLS_X = ['sent_id', 'token_id', 'token']
COLS_Y = 'tipe_afiks'

df_train = df_train_raw[COLS_X + [COLS_Y]].copy()
df_dev   = df_dev_raw  [COLS_X + [COLS_Y]].copy()
df_test  = df_test_raw [COLS_X + [COLS_Y]].copy()

# Encode label
label_encoder = LabelEncoder()
label_encoder.fit(df_train[COLS_Y])

y_train = label_encoder.transform(df_train[COLS_Y])
y_dev   = label_encoder.transform(df_dev  [COLS_Y])
y_test  = label_encoder.transform(df_test [COLS_Y])

NUM_CLASSES = len(label_encoder.classes_)
NUM_LAYERS  = 13    # layer-0 (embedding) + layer 1–12 (transformer)
HIDDEN_SIZE = 768

print("=" * 55)
print("INFO DATASET MORFOLOGI AFIKSASI")
print("=" * 55)
print(f"Kolom X : {COLS_X}")
print(f"Kolom Y : {COLS_Y}")
print(f"Kelas   : {list(label_encoder.classes_)}")
print(f"Train   : {len(df_train):,} token")
print(f"Dev     : {len(df_dev):,} token")
print(f"Test    : {len(df_test):,} token")

INFO DATASET MORFOLOGI AFIKSASI
Kolom X : ['sent_id', 'token_id', 'token']
Kolom Y : tipe_afiks
Kelas   : ['Konfiks', 'Prefiks', 'Sufiks', 'Tidak Berimbuhan']
Train   : 82,963 token
Dev     : 10,676 token
Test    : 10,056 token


### 2. Ekstraksi Hidden States

Yang dilakukan:
1. Mengekstrak hidden states semua layer untuk setiap token

Input:
1. df: DataFrame dengan kolom [sent_id, token_id, token, tipe_afiks]
2. tokenizer, model : model HuggingFace yang sudah di-load

Output:
1. np.array shape (total_token, num_layer, hidden_size) | misal: (9232, 13, 768)
2. Layer 0     = embedding statis (non-contextual)
3. Layer 1–12  = transformer layers (contextual)

In [16]:
# ============================================================
# TAHAP 2: FUNGSI EKSTRAKSI HIDDEN STATES
# ============================================================

def extract_hidden_states(df, tokenizer, model, device, desc="Ekstraksi"):
    model.eval()
    all_hidden  = []
    skipped_cnt = 0

    # Groupby sent_id, sort token berdasarkan token_id
    grouped = df.groupby('sent_id', sort=False)

    for sent_id, group in tqdm(grouped, desc=desc):
        group  = group.sort_values('token_id')
        tokens = group['token'].tolist()

        # Tokenisasi — is_split_into_words=True karena input sudah list kata
        # Satu kata bisa dipecah jadi beberapa subword oleh tokenizer
        encoding = tokenizer(
            tokens,
            is_split_into_words = True,
            return_tensors      = "pt",
            padding             = True,
            truncation          = True,
            max_length          = 512
        )

        input_ids      = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(
                input_ids      = input_ids,
                attention_mask = attention_mask
            )

        hidden_states = outputs.hidden_states

        # ── Mapping subword → token asli ──────────────────────────
        # word_ids() mengembalikan list index kata asli untuk tiap subword
        # Contoh: "membangun" → ["mem", "##bangun"]
        #   word_ids = [None, 0, 0, 0, 1, 1, 2, None]
        #              [CLS] [mem][##b][##u][tok1][tok2] [SEP]
        # Strategi: ambil subword PERTAMA sebagai representasi kata
        word_ids = encoding.word_ids(batch_index=0)

        first_subword_pos = {}
        for pos, word_idx in enumerate(word_ids):
            if word_idx is not None and word_idx not in first_subword_pos:
                first_subword_pos[word_idx] = pos

        for word_idx in range(len(tokens)):
            if word_idx not in first_subword_pos:
                skipped_cnt += 1
                continue

            pos = first_subword_pos[word_idx]

            token_repr = np.array([
                hidden_states[layer_idx][0, pos, :].cpu().numpy()
                for layer_idx in range(len(hidden_states))
            ])

            all_hidden.append(token_repr)

    if skipped_cnt > 0:
        print(f"  ⚠ {skipped_cnt} token dilewati akibat truncation (max_length=512)")

    return np.array(all_hidden) 

### 3. Eksekusi Ekstraksi: IndoBERT

Direktori Hidden States Morfologi Afiksasi:

In [17]:
SAVE_DIR = r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\notebook\Ekstraksi_Hidden_States\Morfologi_Afiksasi"
os.makedirs(SAVE_DIR, exist_ok=True)

Ekstraksi IndoBERT:

In [18]:
# ============================================================
# TAHAP 3: EKSTRAKSI FITUR (INDOBERT) & SIMPAN KE DISK
# ============================================================
# ⚠ Cell ini cukup dijalankan SEKALI.
#   Setelah file .npz tersimpan, langsung lompat ke Tahap 5.
# ============================================================

# ── IndoBERT ──────────────────────────────────────────────────────────────
print("=" * 60)
print("EKSTRAKSI HIDDEN STATES — IndoBERT")
print("=" * 60)

X_train_indobert = extract_hidden_states(
    df_train, tokenizer_indobert, model_indobert, device, "IndoBERT | Train")
X_dev_indobert   = extract_hidden_states(
    df_dev,   tokenizer_indobert, model_indobert, device, "IndoBERT | Dev  ")
X_test_indobert  = extract_hidden_states(
    df_test,  tokenizer_indobert, model_indobert, device, "IndoBERT | Test ")

print(f"\nShape → Train: {X_train_indobert.shape} | "
      f"Dev: {X_dev_indobert.shape} | Test: {X_test_indobert.shape}")

print("Menyimpan IndoBERT ke disk...")
np.savez_compressed(
    os.path.join(SAVE_DIR, "indobert_morph.npz"),
    train   = X_train_indobert,
    dev     = X_dev_indobert,
    test    = X_test_indobert,
    y_train = y_train,
    y_dev   = y_dev,
    y_test  = y_test
)
print(f"✅ Tersimpan → indobert_morph.npz")

del X_train_indobert, X_dev_indobert, X_test_indobert  # bebaskan RAM
print(f"   Lokasi: {SAVE_DIR}")

EKSTRAKSI HIDDEN STATES — IndoBERT


IndoBERT | Train: 100%|██████████| 4476/4476 [02:28<00:00, 30.13it/s]


  ⚠ 2 token dilewati akibat truncation (max_length=512)


IndoBERT | Dev  : 100%|██████████| 559/559 [00:20<00:00, 26.91it/s]
IndoBERT | Test : 100%|██████████| 557/557 [00:18<00:00, 29.71it/s]



Shape → Train: (82961, 13, 768) | Dev: (10676, 13, 768) | Test: (10056, 13, 768)
Menyimpan IndoBERT ke disk...
✅ Tersimpan → indobert_morph.npz
   Lokasi: D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\notebook\Ekstraksi_Hidden_States\Morfologi_Afiksasi


### 4. Eksekusi Ekstraksi: XLM-R

In [19]:
# ============================================================
# TAHAP 4: EKSTRAKSI FITUR (XLM-R) & SIMPAN KE DISK
# ============================================================
# ⚠ Cell ini cukup dijalankan SEKALI.
#   Setelah file .npz tersimpan, langsung lompat ke Tahap 5.
# ============================================================

# ── XLM-R ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("EKSTRAKSI HIDDEN STATES — XLM-R")
print("=" * 60)

X_train_xlmr = extract_hidden_states(
    df_train, tokenizer_xlmr, model_xlmr, device, "XLM-R   | Train")
X_dev_xlmr   = extract_hidden_states(
    df_dev,   tokenizer_xlmr, model_xlmr, device, "XLM-R   | Dev  ")
X_test_xlmr  = extract_hidden_states(
    df_test,  tokenizer_xlmr, model_xlmr, device, "XLM-R   | Test ")

print(f"\nShape → Train: {X_train_xlmr.shape} | "
      f"Dev: {X_dev_xlmr.shape} | Test: {X_test_xlmr.shape}")

print("Menyimpan XLM-R ke disk...")
np.savez_compressed(
    os.path.join(SAVE_DIR, "xlmr_morph.npz"),
    train   = X_train_xlmr,
    dev     = X_dev_xlmr,
    test    = X_test_xlmr,
    y_train = y_train,
    y_dev   = y_dev,
    y_test  = y_test
)
print(f"✅ Tersimpan → xlmr_morph.npz")

del X_train_xlmr, X_dev_xlmr, X_test_xlmr  # bebaskan RAM
print(f"   Lokasi: {SAVE_DIR}")


EKSTRAKSI HIDDEN STATES — XLM-R


XLM-R   | Train: 100%|██████████| 4476/4476 [02:09<00:00, 34.50it/s]


  ⚠ 2 token dilewati akibat truncation (max_length=512)


XLM-R   | Dev  : 100%|██████████| 559/559 [00:16<00:00, 33.27it/s]
XLM-R   | Test : 100%|██████████| 557/557 [00:15<00:00, 35.54it/s]



Shape → Train: (82961, 13, 768) | Dev: (10676, 13, 768) | Test: (10056, 13, 768)
Menyimpan XLM-R ke disk...
✅ Tersimpan → xlmr_morph.npz
   Lokasi: D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\notebook\Ekstraksi_Hidden_States\Morfologi_Afiksasi


### 5. Load Hasil Ekstraksi Hidden States

In [20]:
# ==========================
# TAHAP 5: LOAD DARI DISK  
# ==========================

SAVE_DIR = r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\notebook\Ekstraksi_Hidden_States\Morfologi_Afiksasi"

print("Memuat hidden states dari disk...")
data_indobert    = np.load(os.path.join(SAVE_DIR, "indobert_morph.npz"))
X_train_indobert = data_indobert['train']
X_dev_indobert   = data_indobert['dev']
X_test_indobert  = data_indobert['test']
y_train          = data_indobert['y_train']
y_dev            = data_indobert['y_dev']
y_test           = data_indobert['y_test']

data_xlmr    = np.load(os.path.join(SAVE_DIR, "xlmr_morph.npz"))
X_train_xlmr = data_xlmr['train']
X_dev_xlmr   = data_xlmr['dev']
X_test_xlmr  = data_xlmr['test']

print(f"✅ IndoBERT → Train: {X_train_indobert.shape} | "
      f"Dev: {X_dev_indobert.shape} | Test: {X_test_indobert.shape}")
print(f"✅ XLM-R    → Train: {X_train_xlmr.shape} | "
      f"Dev: {X_dev_xlmr.shape} | Test: {X_test_xlmr.shape}")

Memuat hidden states dari disk...
✅ IndoBERT → Train: (82961, 13, 768) | Dev: (10676, 13, 768) | Test: (10056, 13, 768)
✅ XLM-R    → Train: (82961, 13, 768) | Dev: (10676, 13, 768) | Test: (10056, 13, 768)


### 6. Training Probing Classifier (Linear Classifier - Single Linear Layer)

In [21]:
# ==========================================================
# TAHAP 6: Training Probing Classifier & Visualisasi 
# ==========================================================

# ── Definisi Probing Classifier ───────────────────────────────────────────
class LinearProbe(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.fc(x)


def train_probe(X_train, y_train, X_eval, y_eval,
                hidden_size=768, num_classes=4,
                epochs=10, lr=1e-3, batch_size=64):
    """Melatih probing classifier untuk satu layer. Return (acc, f1)."""
    X_tr = torch.tensor(X_train, dtype=torch.float32)
    y_tr = torch.tensor(y_train, dtype=torch.long)
    X_ev = torch.tensor(X_eval,  dtype=torch.float32)
    y_ev = torch.tensor(y_eval,  dtype=torch.long)

    loader    = DataLoader(TensorDataset(X_tr, y_tr),
                           batch_size=batch_size, shuffle=True)
    probe     = LinearProbe(hidden_size, num_classes).to(device)
    optimizer = optim.Adam(probe.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        probe.train()
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            criterion(probe(X_b), y_b).backward()
            optimizer.step()

    probe.eval()
    with torch.no_grad():
        y_pred = probe(X_ev.to(device)).argmax(dim=1).cpu().numpy()

    acc = accuracy_score(y_ev.numpy(), y_pred)
    f1  = f1_score(y_ev.numpy(), y_pred, average='weighted')
    return acc, f1


# ── Training per Layer ────────────────────────────────────────────────────
results = {
    'indobert': {'acc': [], 'f1': []},
    'xlmr'    : {'acc': [], 'f1': []}
}

print("\n" + "=" * 65)
print("TRAINING PROBING CLASSIFIER PER LAYER — MORFOLOGI AFIKSASI")
print("=" * 65)

for layer_idx in range(NUM_LAYERS):
    acc_i, f1_i = train_probe(
        X_train_indobert[:, layer_idx, :], y_train,
        X_dev_indobert[:,   layer_idx, :], y_dev,
        hidden_size=HIDDEN_SIZE, num_classes=NUM_CLASSES
    )
    acc_x, f1_x = train_probe(
        X_train_xlmr[:, layer_idx, :], y_train,
        X_dev_xlmr[:,   layer_idx, :], y_dev,
        hidden_size=HIDDEN_SIZE, num_classes=NUM_CLASSES
    )
    results['indobert']['acc'].append(acc_i)
    results['indobert']['f1'].append(f1_i)
    results['xlmr']['acc'].append(acc_x)
    results['xlmr']['f1'].append(f1_x)

    print(f"Layer {layer_idx:>2} | "
          f"IndoBERT → Acc: {acc_i:.4f}  F1: {f1_i:.4f} | "
          f"XLM-R   → Acc: {acc_x:.4f}  F1: {f1_x:.4f}")

# ── Visualisasi Probing ───────────────────────────────────────────────────
layer_ticks = [f"L{i}" for i in range(NUM_LAYERS)]

best_i_acc = int(np.argmax(results['indobert']['acc']))
best_x_acc = int(np.argmax(results['xlmr']['acc']))
best_i_f1  = int(np.argmax(results['indobert']['f1']))
best_x_f1  = int(np.argmax(results['xlmr']['f1']))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Probing Classifier — Morfologi Afiksasi | IndoBERT vs XLM-R',
             fontsize=14, fontweight='bold')

for ax, metric, title, b_i, b_x in zip(
    axes,
    ['acc', 'f1'],
    ['Accuracy', 'Weighted F1'],
    [best_i_acc, best_i_f1],
    [best_x_acc, best_x_f1]
):
    ax.plot(layer_ticks, results['indobert'][metric],
            marker='o', linewidth=2, label='IndoBERT', color='#2980b9')
    ax.plot(layer_ticks, results['xlmr'][metric],
            marker='s', linewidth=2, label='XLM-R',   color='#e74c3c')

    # Highlight titik terbaik
    ax.scatter(b_i, results['indobert'][metric][b_i],
               s=150, zorder=5, color='#2980b9',
               edgecolors='black', linewidths=1.5,
               label=f'Best IndoBERT: L{b_i} ({results["indobert"][metric][b_i]:.4f})')
    ax.scatter(b_x, results['xlmr'][metric][b_x],
               s=150, zorder=5, color='#e74c3c', marker='s',
               edgecolors='black', linewidths=1.5,
               label=f'Best XLM-R: L{b_x} ({results["xlmr"][metric][b_x]:.4f})')

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Layer')
    ax.set_ylabel(title)
    ax.set_xticks(range(NUM_LAYERS))
    ax.set_xticklabels(layer_ticks)
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Tabel ringkasan
print("\n=== RINGKASAN HASIL PROBING CLASSIFIER ===")
print(f"{'Layer':<7} {'IB Acc':>8} {'IB F1':>8} {'XLM Acc':>9} {'XLM F1':>9}")
print("-" * 47)
for i in range(NUM_LAYERS):
    tag = " ← best IB"  if i == best_i_f1 else \
          " ← best XLM" if i == best_x_f1 else ""
    print(f"L{i:<6} {results['indobert']['acc'][i]:>8.4f} "
          f"{results['indobert']['f1'][i]:>8.4f} "
          f"{results['xlmr']['acc'][i]:>9.4f} "
          f"{results['xlmr']['f1'][i]:>9.4f}{tag}")


TRAINING PROBING CLASSIFIER PER LAYER — MORFOLOGI AFIKSASI


AssertionError: Size mismatch between tensors

## B. POS Tagging (Syntactic)

## C. Relasi Sintaksis (Syntactic)

## D. Negasi (Semantic)